In [1]:
# Load the CMS Part D spending data into a table and print them out in rows and columns
import duckdb

connection = duckdb.connect()

query = """
SELECT *
FROM read_csv_auto('../data/partd_raw.csv', header = true, all_varchar = true)
"""

data = connection.execute(query).df()

print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))

Number of rows: 14536
Number of columns: 46


In [2]:
# List every column to know what the file contains
for column_name in data.columns:
    print(column_name)

Brnd_Name
Gnrc_Name
Tot_Mftr
Mftr_Name
Tot_Spndng_2020
Tot_Dsg_Unts_2020
Tot_Clms_2020
Tot_Benes_2020
Avg_Spnd_Per_Dsg_Unt_Wghtd_2020
Avg_Spnd_Per_Clm_2020
Avg_Spnd_Per_Bene_2020
Outlier_Flag_2020
Tot_Spndng_2021
Tot_Dsg_Unts_2021
Tot_Clms_2021
Tot_Benes_2021
Avg_Spnd_Per_Dsg_Unt_Wghtd_2021
Avg_Spnd_Per_Clm_2021
Avg_Spnd_Per_Bene_2021
Outlier_Flag_2021
Tot_Spndng_2022
Tot_Dsg_Unts_2022
Tot_Clms_2022
Tot_Benes_2022
Avg_Spnd_Per_Dsg_Unt_Wghtd_2022
Avg_Spnd_Per_Clm_2022
Avg_Spnd_Per_Bene_2022
Outlier_Flag_2022
Tot_Spndng_2023
Tot_Dsg_Unts_2023
Tot_Clms_2023
Tot_Benes_2023
Avg_Spnd_Per_Dsg_Unt_Wghtd_2023
Avg_Spnd_Per_Clm_2023
Avg_Spnd_Per_Bene_2023
Outlier_Flag_2023
Tot_Spndng_2024
Tot_Dsg_Unts_2024
Tot_Clms_2024
Tot_Benes_2024
Avg_Spnd_Per_Dsg_Unt_Wghtd_2024
Avg_Spnd_Per_Clm_2024
Avg_Spnd_Per_Bene_2024
Outlier_Flag_2024
Chg_Avg_Spnd_Per_Dsg_Unt_23_24
CAGR_Avg_Spnd_Per_Dsg_Unt_20_24


In [3]:
# Register data as a table and check which brand and generic drug appear more than once
connection.register("drugs", data)

query = """
SELECT
    Brnd_Name,
    Gnrc_Name,
    COUNT(*) AS row_count
FROM drugs
GROUP BY Brnd_Name, Gnrc_Name
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10
"""

connection.execute(query).df()

,Brnd_Name,Gnrc_Name,row_count
0,Potassium Chloride*,Potassium Chloride,47
1,Gabapentin,Gabapentin,41
2,Levetiracetam*,Levetiracetam,40
3,Acyclovir*,Acyclovir,38
4,Pantoprazole Sodium*,Pantoprazole Sodium,37
5,Famotidine*,Famotidine,34
6,Nystatin*,Nystatin,32
7,Doxycycline Hyclate*,Doxycycline Hyclate,31
8,Ketorolac Tromethamine*,Ketorolac Tromethamine,31
9,Fluoxetine HCl,Fluoxetine HCl,31


In [4]:
# Gives the number of distinct drugs
query = """
SELECT COUNT(*) AS distinct_drugs
FROM drugs
WHERE Mftr_Name = 'Overall'
"""

connection.execute(query).df()

,distinct_drugs
0,3625


In [5]:
# Creates a new table that organizes drug spending, claims, dosage units, and beneficiaries by year from 2020 to 2024

query = """
CREATE OR REPLACE TABLE spending_by_year AS

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2020 AS year,
    TRY_CAST(Tot_Spndng_2020   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2020 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2020     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2020    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2021 AS year,
    TRY_CAST(Tot_Spndng_2021   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2021 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2021     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2021    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2022 AS year,
    TRY_CAST(Tot_Spndng_2022   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2022 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2022     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2022    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2023 AS year,
    TRY_CAST(Tot_Spndng_2023   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2023 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2023     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2023    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2024 AS year,
    TRY_CAST(Tot_Spndng_2024   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2024 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2024     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2024    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'
"""

connection.execute(query)

print("Table created.")

Table created.


In [6]:
# Counts how many rows of drug data are included for each year

query = """
SELECT
    year,
    COUNT(*) AS row_count
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,row_count
0,2020,3625
1,2021,3625
2,2022,3625
3,2023,3625
4,2024,3625


In [7]:
# Calculates total drug spending for each year and shows how many rows have spending data

query = """
SELECT
    year,
    SUM(spending) / 1000000000 AS spending_billions,
    COUNT(*) AS total_rows,
    COUNT(spending) AS rows_with_value
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,spending_billions,total_rows,rows_with_value
0,2020,194.092038,3625,2887
1,2021,213.513349,3625,3054
2,2022,239.779216,3625,3221
3,2023,275.808919,3625,3405
4,2024,288.667842,3625,3625


In [8]:
# Calculates yearly drug spending, dosage units, and the average spending per dosage unit

query = """
SELECT
    year,
    SUM(spending) / 1000000000 AS spending_billions,
    SUM(dosage_units) / 1000000000 AS units_billions,
    SUM(spending) / SUM(dosage_units) AS price_per_unit
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,spending_billions,units_billions,price_per_unit
0,2020,194.092038,110.824788,1.751341
1,2021,213.513349,113.394110,1.882932
2,2022,239.779216,119.426392,2.007757
3,2023,275.808919,124.660472,2.212481
4,2024,288.667842,131.212311,2.200006
